# Stage 11: Final comparison of three systems

All three answer the same 40 eval questions with **Claude Sonnet 5.5 through the API**, using the same system prompt,
answer schema, `get_image` tool and citation checks (`answer.run_answer`). Only the context differs:

| system | context given to the model |
|---|---|
| **RAG, top 10** | dense + BM25 fused, top 30 reranked by MiniLM, best 10 kept |
| **RAG, dynamic cutoff** | top 50 fused reranked by MiniLM; keep those scoring at least the **frozen** cutoff, clamped to 3-15 |
| **Whole handbook** | all 254 articles, ~673k Claude tokens, prompt-cached between questions |

Every answer is graded by **Claude Opus 5.5 through the API** against the reference answers (`judge.py`).

**The dynamic cutoff was frozen before this comparison.** It is stored in `evals/frozen_dynamic_selection.json`
and was chosen in notebook 09b by a rule stated in advance. It must not be changed after seeing these results;
the cell below checks that the frozen file was committed before any of these answers.

**Running:** `python -m handbook_qa.compare answer|judge <systems>` (`src/handbook_qa/compare.py`) made the files in
`artifacts/answers/api/`. It is resumable and records token usage and time per question. This notebook reads
those files; with `RUN_API = True` it calls the same runner to fill in anything missing.

In [1]:
import json
import subprocess

import numpy as np
import pandas as pd

from handbook_qa import compare
from handbook_qa import evaluation as ev
from handbook_qa.config import ROOT, load_settings
from handbook_qa.judge import SCORES
from handbook_qa.retrieval import DynamicSelection, Retriever

RUN_API = False
settings = load_settings()
NAMES = {"rag_top10": "RAG, top 10", "rag_dynamic": "RAG, dynamic cutoff", "full_context": "Whole handbook"}
# List prices, $ per million tokens (Claude API): Sonnet 5.5 in/out 2/10, Opus 5.5 4/20; cache writes 1.25x input,
# cache reads 0.1x input.
PRICES = {"claude-sonnet-5-5": (2.0, 10.0), "claude-opus-5-5": (4.0, 20.0)}
pd.set_option("display.max_colwidth", 110)

# This comparison was run on the first 40 (dev) questions. The answer files now also hold the 60 held-out questions
# from the final face-off (notebook 12), so everything here is filtered to the dev split.
questions = [q for q in ev.load_eval_set(compare.EVAL_SET) if q.split == "dev"]
DEV_IDS = {q.id for q in questions}
selection = DynamicSelection.load(compare.FROZEN)
print("frozen selection:", selection)
print("rule:", json.loads(compare.FROZEN.read_text(encoding="utf-8"))["rule"])

frozen selection: DynamicSelection(cutoff=-3.898, candidates=50, min_keep=3, max_keep=15, model='Xenova/ms-marco-MiniLM-L-6-v2')
rule: Choose the cutoff with the highest mean chunk recall subject to mean chunks kept <= 10 (clamp 3-15 applied per question), over a grid of midpoints between consecutive distinct observed MiniLM scores of the tuning questions (plus one 1.0 below the lowest and one 1.0 above the highest). Break ties by fewer mean chunks, then by the higher cutoff; recalls within 1e-9 count as tied.


## The cutoff was frozen before any of these answers

In [2]:
def first_commit_time(path):
    out = subprocess.run(["git", "log", "--diff-filter=A", "--format=%ct", "--", str(path.relative_to(ROOT))],
                         cwd=ROOT, capture_output=True, text=True).stdout.split()
    return int(out[-1]) if out else None


frozen_at = first_commit_time(compare.FROZEN)
for system in NAMES:
    answers_at = first_commit_time(compare.answers_path(system))
    assert frozen_at and (answers_at is None or frozen_at < answers_at), f"{system} answered before the freeze"
print("OK: frozen file committed at", frozen_at, "- before every system's answers file")

OK: frozen file committed at 1792982183 - before every system's answers file


## Run anything missing (only with RUN_API)

In [3]:
if RUN_API:
    ctx = compare.Context(settings, need_retriever=True)
    for system in NAMES:
        compare.answer_system(ctx, system, workers=1 if system == "full_context" else 4)
    for system in NAMES:
        compare.judge_system(ctx, system, workers=4)

answers = {s: {k: v for k, v in compare.read_rows(compare.answers_path(s)).items() if k in DEV_IDS} for s in NAMES}
judged = {s: {k: v for k, v in compare.read_rows(compare.judgements_path(s)).items() if k in DEV_IDS} for s in NAMES}
for s in NAMES:
    print(f"{NAMES[s]:22} answers {len(answers[s])}/40, judgements {len(judged[s])}/40")

RAG, top 10            answers 40/40, judgements 40/40
RAG, dynamic cutoff    answers 40/40, judgements 40/40
Whole handbook         answers 40/40, judgements 40/40


## Retrieval: what each system put in front of the model

The saved answers record which excerpts the model saw; recall is computed from those.

In [4]:
retriever = Retriever.from_settings(settings)
ev.resolve(questions, retriever.items)
by_id = {q.id: q for q in questions}
rows = []
for s in ("rag_top10", "rag_dynamic"):
    for qid, row in answers[s].items():
        ids = row["excerpt_ids"]
        rows.append({"system": NAMES[s], "id": qid, "chunks": len(ids), "recall": ev.chunk_recall(ids, by_id[qid], len(ids))})
context = pd.DataFrame(rows).groupby("system", sort=False)[["chunks", "recall"]].mean()
context.loc[NAMES["full_context"]] = [254, 1.0]
context.round(3)

C:\Users\shaba\Documents\extra project\paradigm_assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,chunks,recall
system,,
"RAG, top 10",10.000,0.846
"RAG, dynamic cutoff",9.725,0.825
Whole handbook,254.000,1.000


## Answers, citations and judgements

In [5]:
def cost(usage: dict, model: str) -> float:
    price_in, price_out = PRICES[model]
    return (usage.get("input_tokens", 0) * price_in + usage.get("cache_creation_input_tokens", 0) * price_in * 1.25
            + usage.get("cache_read_input_tokens", 0) * price_in * 0.1 + usage.get("output_tokens", 0) * price_out) / 1e6


summary = {}
for s, name in NAMES.items():
    a = pd.DataFrame(answers[s].values())
    j = pd.DataFrame(judged[s].values())
    col = {"questions answered": len(a)}
    if len(a):
        col["citations correct"] = a["citations_ok"].sum() / max(a["citations_total"].sum(), 1)
        col["answers not 'full' coverage"] = int((a["coverage"] != "full").sum())
        col["images fetched"] = int(a["images_viewed"].map(len).sum())
        col["median seconds / answer"] = a["seconds"].median()
        col["input tokens / answer (incl. cache)"] = a["usage"].map(
            lambda u: u["input_tokens"] + u["cache_creation_input_tokens"] + u["cache_read_input_tokens"]).mean()
        col["answer cost, all 40 ($)"] = a["usage"].map(lambda u: cost(u, settings.answer.model)).sum()
    if len(j):
        for dim, values in SCORES.items():
            col[f"judge: {dim}"] = j[dim].map(values).mean()
        col["judge: fully right (all four)"] = ((j["correctness"] == "correct") & (j["completeness"] == "complete")
                                                & (j["groundedness"] == "grounded")
                                                & (j["coverage_honesty"] == "appropriate")).mean()
        col["judge cost, all 40 ($)"] = j["usage"].map(lambda u: cost(u, settings.judge.model)).sum()
    summary[name] = col
comparison = pd.DataFrame(summary)
comparison.round(3)

,"RAG, top 10","RAG, dynamic cutoff",Whole handbook
questions answered,40.000,40.000,40.000
citations correct,0.995,0.990,0.985
answers not 'full' coverage,7.000,7.000,3.000
images fetched,0.000,0.000,1.000
median seconds / answer,4.900,4.800,9.200
input tokens / answer (incl. cache),10332.375,10013.775,690138.450
"answer cost, all 40 ($)",0.968,0.936,7.325
judge: correctness,0.962,0.950,0.988
judge: completeness,0.862,0.825,0.938
judge: groundedness,0.962,0.950,0.900


Judge verdicts, counted:

In [6]:
counts = {}
for s, name in NAMES.items():
    j = pd.DataFrame(judged[s].values())
    if len(j):
        for dim in ["correctness", "completeness", "groundedness", "coverage_honesty", "evidence_in_excerpts"]:
            for value, n in j[dim].value_counts().items():
                counts.setdefault((dim, value), {})[name] = n
pd.DataFrame(counts).T.fillna(0).astype(int)

RAG, top 10  RAG, dynamic cutoff  \
correctness          correct                     38                   37   
                     incorrect                    1                    1   
                     partially_correct            1                    2   
completeness         complete                    32                   29   
                     partial                      5                    8   
                     missing                      3                    3   
groundedness         grounded                    37                   36   
                     minor_unsupported            3                    4   
coverage_honesty     appropriate                 38                   39   
                     underclaimed                 1                    0   
                     overclaimed                  1                    1   
evidence_in_excerpts all                         30                   29   
                     some                         7                    8   
                     none                         3                    3   

                                        Whole handbook  
correctness          correct                        39  
                     incorrect                       0  
                     partially_correct               1  
completeness         complete                       35  
                     partial                         5  
                     missing                         0  
groundedness         grounded                       32  
                     minor_unsupported               8  
coverage_honesty     appropriate                    38  
                     underclaimed                    2  
                     overclaimed                     0  
evidence_in_excerpts all                            40  
                     some                            0  
                     none                            0

Completeness by question type (1 = complete, 0.5 = partial, 0 = missing):

In [7]:
types = {q.id: q.type for q in questions}
by_type = pd.DataFrame({name: pd.DataFrame(judged[s].values()).assign(type=lambda d: d["id"].map(types))
                        .groupby("type")["completeness"].apply(lambda c: c.map(SCORES["completeness"]).mean())
                        for s, name in NAMES.items() if judged[s]})
by_type.insert(0, "questions", pd.Series(types).value_counts())
by_type.round(2)

,questions,"RAG, top 10","RAG, dynamic cutoff",Whole handbook
type,,,,
existing,11,1.00,0.95,0.95
multi-article,12,0.71,0.67,0.92
multi-section,8,1.00,0.94,0.88
reworded,9,0.78,0.78,1.00


## Where the systems differ

In [8]:
per_q = pd.DataFrame({name: {qid: f"{j['correctness'][:7]}/{j['completeness']}" for qid, j in judged[s].items()}
                      for s, name in NAMES.items() if judged[s]})
per_q.insert(0, "question", [by_id[i].question for i in per_q.index])
differ = per_q[per_q.drop(columns="question").nunique(axis=1) > 1]
print(len(differ), "of", len(per_q), "questions get different verdicts across systems")
differ

14 of 40 questions get different verdicts across systems


,question,"RAG, top 10","RAG, dynamic cutoff",Whole handbook
A-06,"If I eventually leave to found my own startup, would PostHog put money into it?",correct/complete,correct/complete,partial/complete
A-09,"I've just been made lead of a small team. Am I now my teammates' manager, who decides what we work on, and...",correct/complete,partial/partial,correct/partial
A-10,"I'm flying abroad for my team's offsite. Which budget should my travel go on, do I need to buy travel insu...",correct/complete,correct/complete,correct/partial
A-11,Someone texted me pretending to be James and asking me to buy gift cards urgently. What should I do?,correct/partial,correct/complete,correct/complete
A-12,"I'm thinking about quitting. How much notice do I give, who do I tell, and do I have to send back my work ...",correct/partial,correct/partial,correct/complete
C1-02,How do we handle refund requests?,correct/complete,correct/partial,correct/partial
C1-04,A big customer asked whether they can pay for PostHog through their Amazon cloud account instead of us inv...,correct/missing,correct/missing,correct/complete
C1-07,"I'm taking two weeks of holiday. How do I stop new leads coming to me while I'm away, and will I get flood...",correct/complete,correct/partial,correct/complete
C1-08,A startup's free credits from our startup program are about to run out. What can I offer them to keep them...,correct/partial,correct/partial,correct/complete
C1-09,I want to let one of my paying customers try another PostHog product for free for a few weeks before they ...,incorre/missing,incorre/missing,correct/complete


In [9]:
for qid in differ.index[:3]:
    print(f"=== {qid}: {by_id[qid].question}\nreference: {by_id[qid].answer}\n")
    for s, name in NAMES.items():
        if qid in judged[s]:
            print(f"--- {name}: {answers[s][qid]['coverage']}\n{answers[s][qid]['answer']}\n"
                  f"judge: {judged[s][qid]['rationale']}\n")

=== A-06: If I eventually leave to found my own startup, would PostHog put money into it?
reference: Yes. If you spend two years at PostHog and then leave to start a new company, PostHog says it will be your first investor and biggest cheerleader.

--- RAG, top 10: full
Yes, in one case. If you spend two years at PostHog and then leave to start a new company, the handbook says PostHog will be your first investor [1]. The excerpts don't say how much PostHog would invest or on what terms.
judge: The answer matches the reference: PostHog will be your first investor if you spend two years there and then leave to start a company. It is backed by a verified quote from the benefits excerpt. The note that the excerpts don't give amounts or terms is accurate and doesn't undercut the 'full' label for the question asked. Leaving out 'biggest cheerleader' is a trivial omission.

--- RAG, dynamic cutoff: partial
Possibly, but only under a narrow condition. The handbook says PostHog will be your fir

## How faithful were the earlier stand-in answers?

Before the API key was connected, Claude Code subagents stood in for the API models (notebook 10). Comparing their
top-10 judgements with the API's shows how far those earlier results can be trusted.

In [10]:
stand_in = {r["id"]: r for r in map(json.loads, (ROOT / "artifacts/answers/judgements.jsonl").read_text(encoding="utf-8").splitlines())}
api = judged["rag_top10"]
shared = sorted(set(stand_in) & set(api))
agree = {dim: np.mean([stand_in[i][dim] == api[i][dim] for i in shared]) for dim in SCORES}
pd.DataFrame({"stand-in mean": {d: np.mean([SCORES[d][stand_in[i][d]] for i in shared]) for d in SCORES},
              "API mean": {d: np.mean([SCORES[d][api[i][d]] for i in shared]) for d in SCORES},
              "same verdict": agree}).round(3)

,stand-in mean,API mean,same verdict
correctness,0.988,0.962,0.950
completeness,0.838,0.862,0.900
groundedness,0.975,0.962,0.875
coverage_honesty,0.975,0.950,0.975
